In [1]:
import requests
import pandas as pd
import time

MAILTO = "Nazrin.Shahpalangova@hsrw.org"
URL = "https://api.openalex.org/works"

SUFFIX = 'AND paper AND coating AND (barrier OR grease OR "water resistance")'

QUERIES = {
    "chitosan":      'chitosan',
    "starch":        'starch',
    "alginate":      'alginate',
    "protein":       '(whey OR soy OR zein OR gluten OR protein)',
    "nanocellulose": '(nanocellulose OR "cellulose nanofibrils" OR "microfibrillated cellulose")',
    "wax":           'wax',
    "lignin_hemi":   '(lignin OR hemicellulose OR xylan)',
}

N_PER_QUERY = 100

In [3]:
def fetch_works(search_text, n):
    params = {
        "search": search_text,
        "filter": "type:article|review",
        "per-page": n,
        "mailto": MAILTO,
    }
    r = requests.get(URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    return data["meta"]["count"], data["results"]

In [4]:
def rebuild_abstract(inv_index):
    if not inv_index:
        return ""
    words = []
    for word, positions in inv_index.items():
        for p in positions:
            words.append((p, word))
    words.sort()
    return " ".join(w for p, w in words)

In [5]:
rows = []

for material, q in QUERIES.items():
    search_text = f"{q} {SUFFIX}"
    total, works = fetch_works(search_text, N_PER_QUERY)
    print(f"{material:15s} ümumi: {total:5d} | götürüldü: {len(works)}")

    for w in works:
        loc = w.get("primary_location") or {}
        src = loc.get("source") or {}
        oa = w.get("open_access") or {}
        rows.append({
            "openalex_id": w["id"],
            "doi": w.get("doi"),
            "title": w.get("title"),
            "year": w.get("publication_year"),
            "type": w.get("type"),
            "journal": src.get("display_name"),
            "cited_by": w.get("cited_by_count"),
            "is_oa": oa.get("is_oa"),
            "oa_url": oa.get("oa_url"),
            "abstract": rebuild_abstract(w.get("abstract_inverted_index")),
            "material_query": material,
        })
    time.sleep(1)

df_raw = pd.DataFrame(rows)
print("\nCəmi sətir:", len(df_raw))

chitosan        ümumi: 33555 | götürüldü: 100
starch          ümumi: 22306 | götürüldü: 100
alginate        ümumi: 18664 | götürüldü: 100
protein         ümumi: 62761 | götürüldü: 100
nanocellulose   ümumi: 10236 | götürüldü: 100
wax             ümumi: 14518 | götürüldü: 100
lignin_hemi     ümumi: 16151 | götürüldü: 100

Cəmi sətir: 700


In [6]:
materials = df_raw.groupby("openalex_id")["material_query"].apply(lambda s: ";".join(sorted(set(s))))

df = df_raw.drop_duplicates("openalex_id").drop(columns="material_query")
df["materials"] = df["openalex_id"].map(materials)

print("Unikal məqalə:", len(df))
print("Abstraktı olan:", (df["abstract"].str.len() > 0).sum())
print("Açıq giriş:", df["is_oa"].sum())
print()
print(df["materials"].value_counts().head(10))

df.to_csv("../data/papers_raw.csv", index=False, encoding="utf-8")
print("\nYadda saxlandı: data/papers_raw.csv")

Unikal məqalə: 566
Abstraktı olan: 468
Açıq giriş: 450

materials
nanocellulose        89
lignin_hemi          85
protein              71
alginate             68
wax                  65
chitosan             57
starch               52
chitosan;protein      6
alginate;chitosan     5
starch;wax            5
Name: count, dtype: int64

Yadda saxlandı: data/papers_raw.csv


In [7]:
for material in QUERIES:
    sub = df_raw[df_raw["material_query"] == material]
    print(f"\n=== {material} ===")
    for idx in [0, 1, 2, 50, 99]:
        print(f"  #{idx+1:3d} | {sub.iloc[idx]['title']}")


=== chitosan ===
  #  1 | Hydrophobization and Antimicrobial Activity of Chitosan and Paper-Based Packaging Material
  #  2 | Barrier and surface properties of chitosan-coated greaseproof paper
  #  3 | Biodegradable plates made of pineapple leaf pulp with biocoatings to improve water resistance
  # 51 | Improved water resistance and air barrier performances of paper coated with polylactic acid/organoclay nanocomposites
  #100 | Recent Trends in the Chemical Modification of Polysaccharides for Food Packaging: A Review

=== starch ===
  #  1 | Organosolv Lignin Barrier Paper Coatings from Waste Biomass Resources
  #  2 | Material properties and water resistance of inorganic–organic polymer coated cellulose paper and nanopaper
  #  3 | Sustainable starch-based barrier coatings for packaging applications
  # 51 | A Review on Reinforcements and Additives in Starch-Based Composites for Food Packaging
  #100 | Silver Nanoparticle-Based Paper Packaging to Combat Black Anther Disease in Orchi

In [8]:
PAPER   = ["paper", "paperboard", "cardboard"]
COAT    = ["coat"]
BARRIER = ["barrier", "grease", "oil resist", "oil-resist", "water resist",
           "water-resist", "wvtr", "water vapor", "water vapour", "cobb",
           "kit test", "kit rating", "hydrophob"]

def has_any(text, keywords):
    return any(k in text for k in keywords)

def is_relevant(row):
    text = f"{row['title'] or ''} {row['abstract'] or ''}".lower()
    return has_any(text, PAPER) and has_any(text, COAT) and has_any(text, BARRIER)

df["relevant"] = df.apply(is_relevant, axis=1)

print("Uyğun:", df["relevant"].sum(), "/", len(df))
print()
per_material = df.assign(m=df["materials"].str.split(";")).explode("m")
print(per_material.groupby("m")["relevant"].agg(["sum", "count"]))

Uyğun: 175 / 566

               sum  count
m                        
alginate        34    100
chitosan        49    100
lignin_hemi     18    100
nanocellulose   25    100
protein         33    100
starch          53    100
wax             50    100


In [9]:
print("=== ÇIXARILANLAR (nümunə) ===")
for t in df[~df["relevant"]]["title"].sample(8, random_state=1):
    print("  ✗", t)

print("\n=== QALANLAR (nümunə) ===")
for t in df[df["relevant"]]["title"].sample(8, random_state=1):
    print("  ✓", t)

no_abs = df["abstract"].str.len() == 0
print("\nÇıxarılanlardan abstraktı olmayan:", (no_abs & ~df["relevant"]).sum())

=== ÇIXARILANLAR (nümunə) ===
  ✗ Factors affecting water resistance of alginate/gellan blend films on paper cups for hot drinks
  ✗ PRESERVATION OF COMMERCIAL FISH BALL QUALITY WITH EDIBLE ANTIOXIDANT-INCORPORATED ZEIN COATINGS
  ✗ Lignocellulosic Materials for the Production of Biofuels, Biochemicals and Biomaterials and Applications of Lignocellulose-Based Polyurethanes: A Review
  ✗ Green-synthesized silver nanoparticle coating on paper for antibacterial and antiviral applications
  ✗ Alginate: From Food Industry to Biomedical Applications and Management of Metabolic Disorders
  ✗ Protein‐based natural antibacterial materials and their applications in food preservation
  ✗ Cellulose Nanopaper: Fabrication, Functionalization, and Applications
  ✗ A Review: Sources, Preparation and Application of Nanocellulose

=== QALANLAR (nümunə) ===
  ✓ Bionanocomposites based on polysaccharides and fibrous clays for packaging applications
  ✓ Packaging-related properties of protein- and chitosan

In [10]:
def is_relevant(row):
    title = (row["title"] or "").lower()
    abstract = (row["abstract"] or "").lower()

    if abstract:
        text = title + " " + abstract
        return has_any(text, PAPER) and has_any(text, COAT) and has_any(text, BARRIER)
    else:
        return has_any(title, PAPER) and (has_any(title, COAT) or has_any(title, BARRIER))

old = df["relevant"].copy()
df["relevant"] = df.apply(is_relevant, axis=1)

print("Uyğun:", df["relevant"].sum(), "/", len(df))
print("\n=== YENİ ƏLAVƏ OLUNANLAR ===")
for t in df[df["relevant"] & ~old]["title"]:
    print("  +", t)

Uyğun: 193 / 566

=== YENİ ƏLAVƏ OLUNANLAR ===
  + Synthesis of hybrid paper sheets with enhanced air barrier and antimicrobial properties for food packaging
  + A facile preparation of chitosan-based oleyl glycidyl ether for waterproof and oil-resistant paper
  + Chitosan/polyethylene glycol diglycidyl ether/aminated lignin composite coating to enhance wet strength of paper as a low-carbon alternative to single-use plastics
  + Study of chitosan with different degrees of acetylation as cardboard paper coating
  + Experimental investigation of novel curdlan/chitosan coatings on packaging paper
  + Development and Characterization of Chitosan and Beeswax–Chitosan Coated Biodegradable Corn Husk and Sugarcane Bagasse-Based Cellulose Paper
  + Fibrous packaging paper made of oil palm fiber with beeswax-chitosan solution to improve water resistance
  + Effects of formation and penetration properties of biodegradable montmorillonite/chitosan nanocomposite film on the barrier of package paper

In [11]:
MANUAL_EXCLUDE = [
    "Green-synthesized silver nanoparticle coating on paper",
    "Nacre-inspired hemicelluloses paper",
]

for phrase in MANUAL_EXCLUDE:
    mask = df["title"].str.contains(phrase, case=False, na=False, regex=False)
    df.loc[mask, "relevant"] = False
    print("Çıxarıldı:", df.loc[mask, "title"].tolist())

screened = df[df["relevant"]].drop(columns="relevant")
screened.to_csv("../data/papers_screened.csv", index=False, encoding="utf-8")

print("\nSon say:", len(screened))
print("Yadda saxlandı: data/papers_screened.csv")

Çıxarıldı: ['Green-synthesized silver nanoparticle coating on paper for antibacterial and antiviral applications']
Çıxarıldı: ['Nacre-inspired hemicelluloses paper with fire retardant and gas barrier properties by self-assembly with bentonite nanosheets']

Son say: 191
Yadda saxlandı: data/papers_screened.csv
